# Python Executor - Attempt #1

**Stage:** execution_failed
**Created:** 2025-09-04 15:17:45

## Error Context

**Error:** Python execution failed: cannot read execution metadata (Python execution error: No module named 'tomopy')

**Full Traceback:**
```
Traceback (most recent call last):
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 161, in _execute_with_subprocess
    raise CodeRuntimeError(
framework.services.python_executor.exceptions.CodeRuntimeError: Python execution error: No module named 'tomopy'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 467, in executor_node
    execution_result = await executor.execute_code(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 57, in execute_code
    return await self._execute_with_subprocess(wrapped_code, execution_folder)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 197, in _execute_with_subprocess
    raise CodeRuntimeError(
framework.services.python_executor.exceptions.CodeRuntimeError: Python execution failed: cannot read execution metadata (Python execution error: No module named 'tomopy')

```

**Execution Stage:** Code execution failed during wrapped code execution

**Debug Information:**
- Error Type: CodeRuntimeError
- Error Message: Python execution failed: cannot read execution metadata (Python execution error: No module named 'tomopy')
- Generated Code Length: 1791 characters




In [ ]:

# Load execution context
from framework.context import load_context
context = load_context('../context.json')


In [ ]:
import numpy as np
from bluesky import RunEngine
from bluesky.plans import scan
from bluesky.preprocessors import baseline_wrapper
from ophyd import Device, Signal
import tomopy
import h5py

class ReconstructionDevice(Device):
    """Custom device for tomographic reconstruction"""
    theta = Signal(name='theta')
    
def tomography_reconstruction_plan(detectors, motor, start, stop, num_points, num_projections=180):
    """
    Bluesky plan for tomographic reconstruction
    
    Parameters:
    - detectors: List of detector devices 
    - motor: Rotation stage motor 
    - start: Initial rotation angle
    - stop: Final rotation angle
    - num_points: Number of rotation points
    - num_projections: Total number of projection angles
    """
    # Generate projection angles
    theta = np.linspace(start, stop, num_projections)
    
    # Baseline wrapped scan for data collection
    @baseline_wrapper
    def _collect_projections():
        yield from scan(detectors, motor, start, stop, num_points)
    
    # Run the scan and collect projection data
    yield from _collect_projections()
    
    # Load collected projection data 
    with h5py.File('test_4/projections.h5', 'r') as f:
        projections = f['data'][:]
    
    # Perform reconstruction using TomoPy
    reconstructed_object = tomopy.recon(projections, theta, algorithm='gridrec')
    
    # Store results 
    results = {
        'reconstructed_object': reconstructed_object,
        'projection_angles': theta
    }
    
    return results

# Create Run Engine
RE = RunEngine({})

# Execute reconstruction plan
results = RE(tomography_reconstruction_plan(
    detectors=[],  # Add actual detector devices
    motor=ReconstructionDevice('rotation_stage'),
    start=0, 
    stop=180, 
    num_points=180
))